# 99 EDA Report (E-35 draft, E-36)

**Data used:** the saved deliverables of notebooks 01 to 08 (`reports/eda/E-*.csv`); run those first.  
Builds the confirmation register (every DOC-02 documented property, confirmed or not confirmed), the answers to Q1-Q16, Q18, Q19 (Q17 stays open until M4 and M7), the nine-section report, and the data-card inputs (E-36). All logic lives in `house_price.eda.report` and `house_price.eda.register` (FR-010).

In [1]:
from IPython.display import Markdown, display

from house_price.eda import EDAContext
from house_price.eda import report

ctx = EDAContext.load()
result = report.build(ctx)

## ADR-fixed discrepancies (must be empty; otherwise stop, DOC-05 M3 task 4)

In [2]:
result.adr_fixed_discrepancies

,property_id,doc02_section,documented_property,evidence,rule,observed,status,adr_fixed


## E-35 Confirmation register

In [3]:
result.register["status"].value_counts()

status
confirmed        59
not confirmed     1
Name: count, dtype: int64

In [4]:
result.register

,property_id,doc02_section,documented_property,evidence,rule,observed,status,adr_fixed
0,P-01,§1.2,"Raw file has 2,930 rows and 82 columns",E-03,exact counts,2930 x 82,confirmed,False
1,P-02,§1.2,79 features; identifiers Id and PID (unique); ...,"E-01, E-03",exact roles; uniqueness and positivity checks ...,"79 features; identifiers ['Id', 'PID'] (unique...",confirmed,False
2,P-03,"§1.2, §14.3",Sales run January 2006 to July 2010; 2010 cove...,E-32,first month of 2006 is 1; last year is 2010 an...,2006-1 to 2010-7,confirmed,False
3,P-04,"§1.2, §9.2","The scope rule removes 5 homes (Ids 1499, 1761...","E-25, E-03",exact Ids and count,"5 removed (Ids [1499, 1761, 1768, 2181, 2182])...",confirmed,True
4,P-05,§1.2,"Development set 2,340 rows; holdout 585 rows (...",E-08,exact counts,"development 2340, holdout 585",confirmed,True
5,P-06,§1.3,The 79 features are 36 numeric and 43 text col...,E-01,exact counts,"79 features: 36 numeric, 43 text",confirmed,False
6,P-07,"§1.4, §5.1","SalePrice ranges from $12,789 to $755,000",E-05,development-set range lies within the document...,"$13,100 to $625,000 (development set)",confirmed,False
7,P-08,"§1.4, §5.1","Median SalePrice about $160,000",E-05,within 10%,"160,000.00 (development set)",confirmed,False
8,P-09,"§1.4, §5.1","Mean SalePrice about $180,800",E-05,within 10%,"180,574.36 (development set)",confirmed,False
9,P-10,§5.1,SalePrice skewness about 1.7,E-05,within 20%,1.66 (development set),confirmed,False


## E-35 Answers to the EDA questions

In [5]:
result.answers

,question,text,answer,evidence
0,Q1,"Does the raw file match the expected shape (2,...",Yes: 2930 rows x 82 columns; all 82 columns ca...,"E-01, E-03"
1,Q2,Are all category values within the data dictio...,Yes: no observed value falls outside the allow...,E-02
2,Q3,Are Id and PID unique and is every SalePrice p...,Id unique: True; PID unique: True; SalePrice >...,E-03
3,Q4,Does the CSV parser change missing-value count...,"Yes, for MasVnrType (23 -> 1775). The explicit...",E-04
4,Q5,"How skewed and heavy-tailed is SalePrice, and ...",Skewness 1.66 and excess kurtosis 4.23 in doll...,"E-05, E-06"
5,Q6,Is residual spread larger for expensive homes ...,Yes: the residual SD in the top fitted-value d...,E-07
6,Q7,"For each column with NA, does it mean 'absent'...","8 columns are purely absent, 8 purely unknown ...","E-09, E-10"
7,Q8,Are 'absent' patterns internally consistent?,"Largely: the garage, basement, and masonry abs...",E-11
8,Q9,Does missingness in LotFrontage relate to price?,Rows with LotFrontage missing (382) have a med...,E-12
9,Q10,Which numeric features are strongly skewed or ...,"Skewness > 1: LotFrontage, LotArea, MasVnrArea...","E-13, E-14"


## E-36 Data-card inputs

In [6]:
result.data_card_inputs

,issue,status,evidence,detail
0,Parsing hazard: MasVnrType literal 'None',confirmed,E-04,"Default parsing would report 1,775 missing ins..."
1,basement_exposure_missing_with_basement: Basem...,confirmed,E-11,"3 row(s): Ids 67, 1797, 2780"
2,basement_fintype2_missing_with_area: BsmtFinSF...,confirmed,E-11,1 row(s): Ids 445
3,basement_areas_unrecorded: Basement square-foo...,confirmed,E-11,1 row(s): Ids 1342
4,garage_partially_recorded: A garage type is re...,confirmed,E-11,"2 row(s): Ids 1357, 2237"
5,garage_built_after_sale: GarageYrBlt is later ...,confirmed,E-11,"2 row(s): Ids 2181, 2261"
6,house_built_after_sale: YearBuilt is later tha...,confirmed,E-11,1 row(s): Ids 2181
7,remodel_before_built: YearRemodAdd is earlier ...,confirmed,E-11,1 row(s): Ids 851
8,masvnr_none_with_area: MasVnrType is 'None' bu...,confirmed,E-11,"7 row(s): Ids 364, 404, 442, 1862, 1914, 2004,..."
9,remodel_year_at_floor: YearRemodAdd = 1950 for...,confirmed,E-11,339 row(s)


## E-35 Report

In [7]:
display(Markdown(result.markdown))

# E-35 EDA Report (draft)

Generated by `python -m house_price.eda` from the saved deliverables in `reports/eda/` and `reports/figures/eda/`. Target relationships use the development set only; the raw file is used for target-free profiling and the ADR-06 scope review (FR-009). Q17 stays open until M4 and M7.

## 1. Executive Summary

**Observations**

- 59 of 60 documented properties confirmed; 1 not confirmed.
- Data integrity (Q1-Q3): Yes: 2930 rows x 82 columns; all 82 columns carry their schema names and 82 of 82 load with the declared dtype. Id unique: True; PID unique: True; SalePrice > 0: True; raw-file hash confirmed: True.
- Target (Q5): SalePrice is right-skewed and log1p makes it near-symmetric. Skewness 1.66 and excess kurtosis 4.23 in dollars; 0.02 and 1.17 after log1p (development set).
- Price drivers (Q11): size and quality dominate. Top by |Spearman|: OverallQual, GrLivArea, GarageCars, YearBuilt, GarageArea.
- Missing values (Q7): 96.3% of 13997 missing cells absent; the genuinely unknown values are LotFrontage, Electrical, and a few recorded anomaly rows.
- Scope (Q15): 5 homes (Ids 1499, 1761, 1768, 2181, 2182). The 3 partial sales sit -8.8, -7.1, -6.2 SD below the in-scope size trend; the others are near-top-priced homes. Removing them changes the log-price slope by +5.9%.

**Evidence**

- E-35_confirmation_register.csv
- E-35_question_answers.csv

**Implications for later milestones**

- M4: implement the semantic filler, engineered features, and ordinal map as designed; the evidence supports them.
- M5: log target wrapper, Yeo-Johnson on skewed numerics, fitted imputation with missing indicators, one-hot/ordinal encoding with unseen-category handling.

## 2. Dataset Overview

**Observations**

- Yes: 2930 rows x 82 columns; all 82 columns carry their schema names and 82 of 82 load with the declared dtype.
- Yes: no observed value falls outside the allowed lists (0 columns with out-of-list values). The file uses its own spellings for some codes (MSZoning, Neighborhood, BldgType, Exterior2nd, SaleType), which the allowed lists include.
- Id unique: True; PID unique: True; SalePrice > 0: True; raw-file hash confirmed: True.
- Yes, for MasVnrType (23 -> 1775). The explicit token contract (DN-18) keeps the counts stable.
- Split balance: every price decile differs by at most 0.13 pp between the sets (tolerance 2 pp).

**Evidence**

- E-01_shape_schema.csv
- E-02_allowed_values_audit.csv
- E-02_value_counts.csv
- E-03_key_integrity.csv
- E-04_parsing_comparison.csv
- E-08_split_balance.csv

**Implications for later milestones**

- The ingestion contract needs no change; allowed values already include the file spellings, so the API accepts exactly the values training saw (M11).
- Keep the explicit missing-token contract (DN-18) everywhere data is parsed (CLI, M11).

## 3. Target Analysis

**Observations**

- Skewness 1.66 and excess kurtosis 4.23 in dollars; 0.02 and 1.17 after log1p (development set).
- Yes: the residual SD in the top fitted-value decile is 2.80x the bottom decile in dollars, versus 0.75x on the log scale.
- Mean exceeds median, the signature of right skew; Q-Q plots show the raw tail departing from normality and the log scale close to the reference line.

**Evidence**

- E-05_target_statistics.csv
- figures/eda/E-06_target_distribution.png
- E-07_residual_spread.csv
- figures/eda/E-07_heteroscedasticity.png

**Implications for later milestones**

- Train on log1p(SalePrice) via TransformedTargetRegressor and select on log-RMSE (ADR-02, ADR-12): M5, M6.
- Stratify folds on binned log price (DN-01): M6.

## 4. Missing-Value Analysis

**Observations**

- 8 columns are purely absent, 8 purely unknown (LotFrontage, Electrical, and fields of unrecorded rows), 11 mixed: absent except for the documented anomaly rows.
- Rows with LotFrontage missing (382) have a median log price +0.091 (about +9% in dollars) relative to recorded rows (1958), so missingness carries some price signal; the missing-indicator column lets the model use it.
- Largest gaps: PoolQC 99.6%, MiscFeature 96.4%, Alley 93.2%, Fence 80.5%, FireplaceQu 48.5%, LotFrontage 16.7%.

**Evidence**

- E-09_missing_values.csv
- figures/eda/E-10_missingness.png
- E-12_lotfrontage_missingness.csv
- figures/eda/E-12_lotfrontage_missingness.png

**Implications for later milestones**

- M4: Layer 1 semantic filler ('None'/0) for the absent-feature columns; GarageYrBlt replaced by HasGarage/GarageAge.
- M5: Layer 2 median + missing indicator for LotFrontage, most frequent for Electrical, fitted inside the pipeline.

## 5. Numerical Feature Analysis

**Observations**

- Skewness > 1: LotFrontage, LotArea, MasVnrArea, BsmtFinSF2, LowQualFinSF, BsmtHalfBath, KitchenAbvGr, WoodDeckSF, OpenPorchSF, EnclosedPorch, 3SsnPorch, ScreenPorch, PoolArea, MiscVal. At least 50% zeros: MasVnrArea, BsmtFinSF2, 2ndFlrSF, LowQualFinSF, BsmtFullBath, BsmtHalfBath, HalfBath, WoodDeckSF, EnclosedPorch, 3SsnPorch, ScreenPorch, PoolArea, MiscVal.
- Values beyond the 99th percentile exist in all 10 key features; all are within the schema's physical ranges. Features whose extremes include rows flagged by a consistency rule: TotalBsmtSF. No statistical outlier removal is warranted (ADR-06).
- Near-constant (top value >= 90%): LowQualFinSF, BsmtHalfBath, KitchenAbvGr, 3SsnPorch, ScreenPorch, PoolArea, MiscVal.

**Evidence**

- E-13_numeric_profile.csv
- figures/eda/E-14_numeric_distributions.png
- E-17_overallqual_price_profile.csv
- figures/eda/E-17_top_predictor_scatter.png
- E-18_numeric_extremes.csv

**Implications for later milestones**

- M5: Yeo-Johnson power transform and scaling in the linear branch; none in the tree branch.
- M4: presence flags (HasPool, Has2ndFlr, ...) for zero-inflated columns; aggregates TotalSF, TotalBath, TotalPorchSF.
- No statistical outlier removal beyond the scope rule (ADR-06).

## 6. Categorical Feature Analysis

**Observations**

- Top level covers >= 95% of rows: Condition2, Heating, RoofMatl, PoolQC, MiscFeature, LandSlope, Utilities, Street. 104 categories in 34 features fall under 1% of development rows.
- Monotonic: ExterQual, BsmtQual, BsmtCond, HeatingQC, KitchenQual, FireplaceQu, GarageQual. Not monotonic: ExterCond, GarageCond.
- Most informative categoricals (eta²): Neighborhood 0.59, BsmtQual 0.49, ExterQual 0.47, KitchenQual 0.45, GarageFinish 0.38, MSSubClass 0.33.
- Least informative categoricals (eta²): RoofMatl, PoolQC, MiscFeature, LandSlope, Utilities.

**Evidence**

- E-19_categorical_profile.csv
- E-19_foundation_year_built.csv
- E-20_rare_categories.csv
- E-21_category_importance.csv
- E-21_price_by_category.csv
- figures/eda/E-21_price_by_category_*.png
- E-22_neighborhoods.csv
- figures/eda/E-22_neighborhoods.png
- E-23_ordinal_monotonicity.csv
- figures/eda/E-23_ordinal_monotonicity.png

**Implications for later milestones**

- M4: ordinal map (None=0 ... Ex=5) for the ten Po-Ex scales; MSSubClass cast to string.
- M5: one-hot (linear) / ordinal encoding (tree) with unseen-category handling for rare levels; no target encoding.
- Non-monotonic condition scales are a known limitation of the linear branch; trees can split them freely (DOC-02 §8.6).

## 7. Correlation Analysis

**Observations**

- Top by |Spearman|: OverallQual, GrLivArea, GarageCars, YearBuilt, GarageArea. Strongly correlated pairs (|r| >= 0.7): GarageCars-GarageArea (0.89), YearBuilt-GarageYrBlt (0.83), GrLivArea-TotRmsAbvGrd (0.81), TotalBsmtSF-1stFlrSF (0.79).
- Mostly monotonic and close to linear for the main drivers: for the top 8 predictors Pearson and Spearman differ by 0.031 on average. Step-like effects appear in the categorical and ordinal scales (E-21, E-23) and in OverallQual's convex dollar profile (E-17). Engineered-feature evidence (E-28) completes this answer in M4.
- Least informative numerics (|Spearman|): BsmtFinSF2, BsmtHalfBath, YrSold, 3SsnPorch, PoolArea.

**Evidence**

- E-15_target_correlation.csv
- E-16_correlated_pairs.csv
- figures/eda/E-16_correlation_heatmap.png

**Implications for later milestones**

- M6-M8: regularised linear models (Ridge shares weight, Lasso selects) for the collinear size and garage features; OverallQual + GrLivArea form the heuristic baseline.
- M4: GarageYrBlt's overlap with YearBuilt supports replacing it with GarageAge/HasGarage.

## 8. Anomaly Review

**Observations**

- Largely: the garage, basement, and masonry absence patterns hold, with these flagged exceptions: garage_partially_recorded (Ids 1357, 2237), basement_exposure_missing_with_basement (Ids 67, 1797, 2780), basement_fintype2_missing_with_area (Ids 445), basement_areas_unrecorded (Ids 1342), masvnr_none_with_area (Ids 364, 404, 442, 1862, 1914, 2004, 2529), garage_built_after_sale (Ids 2181, 2261), remodel_before_built (Ids 851).
- 5 homes (Ids 1499, 1761, 1768, 2181, 2182). The 3 partial sales sit -8.8, -7.1, -6.2 SD below the in-scope size trend; the others are near-top-priced homes. Removing them changes the log-price slope by +5.9%.

**Evidence**

- E-11_consistency_checks.csv
- E-11_anomaly_rows.csv
- figures/eda/E-24_scope_rule.png
- E-25_out_of_scope_homes.csv
- E-26_leverage_comparison.csv
- reports/data_validation/quality_flags.csv (M2)

**Implications for later milestones**

- The anomalies are handled by the approved column-wise rules (DOC-02 §6.6): no special code path in M4; recorded in the data card.
- The scope rule (GrLivArea <= 4000) stays as fixed by ADR-06; the API flags larger homes out_of_domain (M11).

## 9. Final EDA Findings

**Observations**

- Not known at listing: SaleType, SaleCondition (excluded; rows kept). Identifiers Id and PID are excluded as non-attributes.
- Sales per year: 2006: 491, 2007: 564, 2008: 490, 2009: 517, 2010: 278 (2010 covers months 1-7). Median log price varies by 0.083 across years. Sales peak in month 6.
- 59 of 60 documented properties confirmed; 1 not confirmed.

**Evidence**

- E-31_leakage_review.csv
- E-31_sale_outcome_context.csv
- E-32_sale_years.csv
- E-33_price_by_year.csv
- E-34_seasonality.csv
- E-35_confirmation_register.csv
- E-36_data_card_inputs.csv

**Implications for later milestones**

- M5: SaleType, SaleCondition, Id, and PID are not model inputs (77-column schema).
- M9: the temporal diagnostic (2006-2009 -> 2010) uses a partial 2010 (January-July).
- Q17 (engineered features) is answered in M4 (E-27, E-28) and M7 (E-30).

## Questions Q1-Q19

| question | answer | evidence |
|---|---|---|
| Q1 | Yes: 2930 rows x 82 columns; all 82 columns carry their schema names and 82 of 82 load with the declared dtype. | E-01, E-03 |
| Q2 | Yes: no observed value falls outside the allowed lists (0 columns with out-of-list values). The file uses its own spellings for some codes (MSZoning, Neighborhood, BldgType, Exterior2nd, SaleType), which the allowed lists include. | E-02 |
| Q3 | Id unique: True; PID unique: True; SalePrice > 0: True; raw-file hash confirmed: True. | E-03 |
| Q4 | Yes, for MasVnrType (23 -> 1775). The explicit token contract (DN-18) keeps the counts stable. | E-04 |
| Q5 | Skewness 1.66 and excess kurtosis 4.23 in dollars; 0.02 and 1.17 after log1p (development set). | E-05, E-06 |
| Q6 | Yes: the residual SD in the top fitted-value decile is 2.80x the bottom decile in dollars, versus 0.75x on the log scale. | E-07 |
| Q7 | 8 columns are purely absent, 8 purely unknown (LotFrontage, Electrical, and fields of unrecorded rows), 11 mixed: absent except for the documented anomaly rows. | E-09, E-10 |
| Q8 | Largely: the garage, basement, and masonry absence patterns hold, with these flagged exceptions: garage_partially_recorded (Ids 1357, 2237), basement_exposure_missing_with_basement (Ids 67, 1797, 2780), basement_fintype2_missing_with_area (Ids 445), basement_areas_unrecorded (Ids 1342), masvnr_none_with_area (Ids 364, 404, 442, 1862, 1914, 2004, 2529), garage_built_after_sale (Ids 2181, 2261), remodel_before_built (Ids 851). | E-11 |
| Q9 | Rows with LotFrontage missing (382) have a median log price +0.091 (about +9% in dollars) relative to recorded rows (1958), so missingness carries some price signal; the missing-indicator column lets the model use it. | E-12 |
| Q10 | Skewness > 1: LotFrontage, LotArea, MasVnrArea, BsmtFinSF2, LowQualFinSF, BsmtHalfBath, KitchenAbvGr, WoodDeckSF, OpenPorchSF, EnclosedPorch, 3SsnPorch, ScreenPorch, PoolArea, MiscVal. At least 50% zeros: MasVnrArea, BsmtFinSF2, 2ndFlrSF, LowQualFinSF, BsmtFullBath, BsmtHalfBath, HalfBath, WoodDeckSF, EnclosedPorch, 3SsnPorch, ScreenPorch, PoolArea, MiscVal. | E-13, E-14 |
| Q11 | Top by , Spearman, : OverallQual, GrLivArea, GarageCars, YearBuilt, GarageArea. Strongly correlated pairs (, r,  >= 0.7): GarageCars-GarageArea (0.89), YearBuilt-GarageYrBlt (0.83), GrLivArea-TotRmsAbvGrd (0.81), TotalBsmtSF-1stFlrSF (0.79). | E-15, E-16, E-17 |
| Q12 | Top level covers >= 95% of rows: Condition2, Heating, RoofMatl, PoolQC, MiscFeature, LandSlope, Utilities, Street. 104 categories in 34 features fall under 1% of development rows. | E-19, E-20 |
| Q13 | Monotonic: ExterQual, BsmtQual, BsmtCond, HeatingQC, KitchenQual, FireplaceQu, GarageQual. Not monotonic: ExterCond, GarageCond. | E-23 |
| Q14 | Mostly monotonic and close to linear for the main drivers: for the top 8 predictors Pearson and Spearman differ by 0.031 on average. Step-like effects appear in the categorical and ordinal scales (E-21, E-23) and in OverallQual's convex dollar profile (E-17). Engineered-feature evidence (E-28) completes this answer in M4. | E-17, E-21 |
| Q15 | 5 homes (Ids 1499, 1761, 1768, 2181, 2182). The 3 partial sales sit -8.8, -7.1, -6.2 SD below the in-scope size trend; the others are near-top-priced homes. Removing them changes the log-price slope by +5.9%. | E-24, E-25, E-26 |
| Q16 | Values beyond the 99th percentile exist in all 10 key features; all are within the schema's physical ranges. Features whose extremes include rows flagged by a consistency rule: TotalBsmtSF. No statistical outlier removal is warranted (ADR-06). | E-18 |
| Q17 | Open: answered in M4 (E-27, E-28) and M7 (ablation, E-30). | E-27, E-28, E-30 |
| Q18 | Not known at listing: SaleType, SaleCondition (excluded; rows kept). Identifiers Id and PID are excluded as non-attributes. | E-31 |
| Q19 | Sales per year: 2006: 491, 2007: 564, 2008: 490, 2009: 517, 2010: 278 (2010 covers months 1-7). Median log price varies by 0.083 across years. Sales peak in month 6. | E-32, E-33, E-34 |

## Confirmation Register

| property_id | doc02_section | documented_property | observed | status |
|---|---|---|---|---|
| P-01 | §1.2 | Raw file has 2,930 rows and 82 columns | 2930 x 82 | confirmed |
| P-02 | §1.2 | 79 features; identifiers Id and PID (unique); target SalePrice (> 0) | 79 features; identifiers ['Id', 'PID'] (unique: True); target ['SalePrice'] (> 0: True) | confirmed |
| P-03 | §1.2, §14.3 | Sales run January 2006 to July 2010; 2010 covers January-July only | 2006-1 to 2010-7 | confirmed |
| P-04 | §1.2, §9.2 | The scope rule removes 5 homes (Ids 1499, 1761, 1768, 2181, 2182), leaving 2,925 rows | 5 removed (Ids [1499, 1761, 1768, 2181, 2182]); 2925 in scope | confirmed |
| P-05 | §1.2 | Development set 2,340 rows; holdout 585 rows (80/20) | development 2340, holdout 585 | confirmed |
| P-06 | §1.3 | The 79 features are 36 numeric and 43 text columns | 79 features: 36 numeric, 43 text | confirmed |
| P-07 | §1.4, §5.1 | SalePrice ranges from $12,789 to $755,000 | $13,100 to $625,000 (development set) | confirmed |
| P-08 | §1.4, §5.1 | Median SalePrice about $160,000 | 160,000.00 (development set) | confirmed |
| P-09 | §1.4, §5.1 | Mean SalePrice about $180,800 | 180,574.36 (development set) | confirmed |
| P-10 | §5.1 | SalePrice skewness about 1.7 | 1.66 (development set) | confirmed |
| P-11 | §5.1 | SalePrice excess kurtosis about 5.1 | 4.23 (development set) | confirmed |
| P-12 | §5.1 | Mean above median: right skew | mean $180,574 vs median $160,000 | confirmed |
| P-13 | §5.3 | log1p reduces skewness to about 0 (near symmetric) | skewness 1.66 -> 0.02 | confirmed |
| P-14 | §5.2 | Error spread grows with price on the raw scale and is much more constant on the log scale | dollar residual SD top/bottom decile ratio 2.80; max/min SD across deciles: dollars 3.77, log 1.59 | confirmed |
| P-15 | §2.2, §3.5 | OverallQual and GrLivArea are the two strongest correlates of price, OverallQual first | rank 1 OverallQual, rank 2 GrLivArea (, Spearman,  with log price) | confirmed |
| P-16 | §3.5 | OverallQual's effect is increasing and convex in dollars and closer to linear in log space | linear-fit R² of median price by level: dollars 0.916, log 0.997; upper-half dollar steps larger: True | confirmed |
| P-17 | §3.5 | OverallCond relates to price more weakly than OverallQual | , Spearman,  OverallCond 0.156, OverallQual 0.812 | confirmed |
| P-18 | §3.1 | Condition2 is almost always Norm | Norm share 99.0% (raw file) | confirmed |
| P-19 | §3.2, §7.1 | LotArea is heavily right-skewed | skewness 12.90 | confirmed |
| P-20 | §3.2, §6.3 | LotFrontage is missing in 490 rows (about 17%) and is genuinely unknown | 490 missing (16.7%), unknown | confirmed |
| P-21 | §3.2 | Alley is mostly NA (no alley access) | 93.2% missing | confirmed |
| P-22 | §3.2, §8.3 | Street is almost always paved (12 gravel streets) | 12 gravel | confirmed |
| P-23 | §3.4 | GrLivArea equals 1stFlrSF + 2ndFlrSF + LowQualFinSF | rule grlivarea_components_differ: clear | confirmed |
| P-24 | §3.4, §7.3 | GrLivArea and TotRmsAbvGrd are strongly correlated | Pearson r = 0.810 | confirmed |
| P-25 | §3.6, §10.5 | YearRemodAdd has a recording floor of 1950 | minimum 1950; 339 pre-1950 houses at 1950 (raw file) | confirmed |
| P-26 | §3.7, §6.6 | 79 houses without a basement have NA in every basement categorical and TotalBsmtSF = 0 | 79 absent rows; basement_absent_with_area clear | confirmed |
| P-27 | §3.8, §6.6 | 157 houses without a garage: garage categoricals and GarageYrBlt NA, GarageArea and GarageCars 0 | GarageType 157 missing, 157 absent; garage_absent_with_area clear | confirmed |
| P-28 | §3.8, §7.3 | GarageCars and GarageArea are strongly correlated | Pearson r = 0.895 | confirmed |
| P-29 | §3.9 | RoofMatl is dominated by standard composite shingle | CompShg share 98.5% (raw file) | confirmed |
| P-30 | §3.9 | Foundation acts as an age proxy: poured concrete is common in newer homes | latest median YearBuilt: PConc (2002) | confirmed |
| P-31 | §3.10, §8.3 | Utilities: all but three houses have all public utilities | 3 non-AllPub | confirmed |
| P-32 | §3.10 | Heating is dominated by gas forced-air | GasA share 98.5% (raw file) | confirmed |
| P-33 | §3.10 | Houses without central air tend to be cheaper | median log price N 11.518, Y 12.026 | confirmed |
| P-34 | §3.10, §6.3 | Electrical has a single genuinely unknown value | 1 missing, unknown | confirmed |
| P-35 | §3.11, §6.2 | FireplaceQu is NA for 1,422 houses without a fireplace | 1422 missing, 1422 with Fireplaces = 0 | confirmed |
| P-36 | §3.12, §7.1 | Outdoor and amenity areas are zero-inflated (porches, 2ndFlrSF, MasVnrArea, PoolArea) | OpenPorchSF 45.2%, EnclosedPorch 84.2%, 3SsnPorch 98.7%, ScreenPorch 91.1%, 2ndFlrSF 57.4%, MasVnrArea 60.1%, PoolArea 99.6% | confirmed |
| P-37 | §3.12 | Only 13 houses have a pool; PoolQC is about 99.6% NA | 2917 missing (99.56%), 2917 with PoolArea = 0 | confirmed |
| P-38 | §3.13 | MiscFeature is about 96% NA and MiscVal is almost always zero | MiscFeature 96.4% missing; MiscVal zero 96.5% | confirmed |
| P-39 | §6.2 | Per-column missing counts of the feature-absent columns (§6.2 table) | all 17 counts match | confirmed |
| P-40 | §6.5 | MasVnrType has 23 missing values and 1,752 literal 'None'; default parsing would report 1,775 missing | explicit 23, default 1775, 'None' 1752 | confirmed |
| P-41 | §6.6 | Basement anomalies: BsmtExposure missing with a basement (Ids 67, 1797, 2780); BsmtFinType2 missing with finished area (Id 445) | basement_exposure_missing_with_basement: [67, 1797, 2780]; basement_fintype2_missing_with_area: [445] | confirmed |
| P-42 | §6.6 | Related unrecorded rows: Id 1342 (all basement fields); Ids 1357 and 2237 (garage fields) | basement_areas_unrecorded: [1342]; garage_partially_recorded: [1357, 2237] | confirmed |
| P-43 | §6.6 | MasVnrType and MasVnrArea are missing together | rule masvnr_type_area_missing_mismatch: clear | confirmed |
| P-44 | §7.1 | Strong right skew in LotArea, GrLivArea, MasVnrArea, MiscVal, PoolArea, LowQualFinSF, 3SsnPorch | LotArea 12.9, GrLivArea 0.9, MasVnrArea 2.4, MiscVal 21.4, PoolArea 18.2, LowQualFinSF 12.8, 3SsnPorch 11.3 | not confirmed |
| P-45 | §7.2 | Near-constant numeric features: PoolArea, 3SsnPorch, LowQualFinSF, MiscVal, KitchenAbvGr | PoolArea 99.6%, 3SsnPorch 98.7%, LowQualFinSF 98.6%, MiscVal 96.5%, KitchenAbvGr 95.1% | confirmed |
| P-46 | §7.3 | Strongest numeric correlates: OverallQual, GrLivArea, GarageCars, GarageArea, TotalBsmtSF, 1stFlrSF, FullBath, TotRmsAbvGrd, YearBuilt, YearRemodAdd | OverallQual #1, GrLivArea #2, GarageCars #3, GarageArea #5, TotalBsmtSF #9, 1stFlrSF #10, FullBath #6, TotRmsAbvGrd #12, YearBuilt #4, YearRemodAdd #8 | confirmed |
| P-47 | §7.3 | TotalBsmtSF and 1stFlrSF are strongly correlated | Pearson r = 0.787 | confirmed |
| P-48 | §7.3 | YearBuilt and GarageYrBlt are strongly correlated | Pearson r = 0.833 | confirmed |
| P-49 | §8.2 | Cardinality: Neighborhood 28, Exterior2nd 17, MSSubClass 16, Exterior1st 16, Condition1 9, SaleType 10, Condition2/HouseStyle/RoofMatl 8 | Neighborhood 28, Exterior2nd 17, MSSubClass 16, Exterior1st 16, Condition1 9, SaleType 10, Condition2 8, HouseStyle 8, RoofMatl 8 | confirmed |
| P-50 | §8.2 | Most other categoricals have between 2 and 7 levels | 34 of 35 other categoricals have 2-7 levels | confirmed |
| P-51 | §8.3 | The smallest neighborhood has 1 sale | smallest: 1 sale(s) | confirmed |
| P-52 | §8.5, §3.1 | Neighborhood levels have strongly different price levels | range 1.30 log units | confirmed |
| P-53 | §8.6 | Quality scales (ExterQual, KitchenQual, BsmtQual) rise clearly with price | ExterQual: monotonic, spread 1.41; KitchenQual: monotonic, spread 1.13; BsmtQual: monotonic, spread 1.15 | confirmed |
| P-54 | §8.6 | Condition scales (ExterCond, GarageCond) show a weaker pattern | ExterCond: not monotonic, spread 0.36; GarageCond: not monotonic, spread 0.53 | confirmed |
| P-55 | §9.2 | 3 of the out-of-scope homes are partial sales priced far below their size trend | 3 partial sales, residuals -8.8, -7.1, -6.2 SD | confirmed |
| P-56 | §9.2 | The other 2 out-of-scope homes are among the most expensive | 2 homes at percentiles 99.97, 100.00 | confirmed |
| P-57 | §9.2 | The out-of-scope homes flatten the size-price slope | slope per 1,000 sq ft: all 0.561, in scope 0.594 | confirmed |
| P-58 | §11.4 | New-construction sales are associated with higher prices | highest: New (+0.46 log vs overall) | confirmed |
| P-59 | §11.5 | Abnormal and family sales tend to sell below market value | Abnorml 11.748, Family 11.822, Normal 11.977 | confirmed |
| P-60 | §1.5, §15.1 | Most missing values mean 'feature absent', not 'unknown' | 96.3% of 13997 missing cells absent | confirmed |

## Data Card Inputs (E-36)

| issue | status | evidence | detail |
|---|---|---|---|
| Parsing hazard: MasVnrType literal 'None' | confirmed | E-04 | Default parsing would report 1,775 missing instead of 23 |
| basement_exposure_missing_with_basement: Basement exists (BsmtQual recorded) but BsmtExposure is missing (DOC-02 §6.6). | confirmed | E-11 | 3 row(s): Ids 67, 1797, 2780 |
| basement_fintype2_missing_with_area: BsmtFinSF2 > 0 but BsmtFinType2 is missing (DOC-02 §6.6). | confirmed | E-11 | 1 row(s): Ids 445 |
| basement_areas_unrecorded: Basement square-footage fields are missing (unknown, not zero). | confirmed | E-11 | 1 row(s): Ids 1342 |
| garage_partially_recorded: A garage type is recorded but finish/quality/condition/year is missing (unknown values inside 'feature absent' columns). | confirmed | E-11 | 2 row(s): Ids 1357, 2237 |
| garage_built_after_sale: GarageYrBlt is later than YrSold (e.g. the documented 2207 typo). | confirmed | E-11 | 2 row(s): Ids 2181, 2261 |
| house_built_after_sale: YearBuilt is later than YrSold (sold before completion). | confirmed | E-11 | 1 row(s): Ids 2181 |
| remodel_before_built: YearRemodAdd is earlier than YearBuilt. | confirmed | E-11 | 1 row(s): Ids 851 |
| masvnr_none_with_area: MasVnrType is 'None' but MasVnrArea > 0. | confirmed | E-11 | 7 row(s): Ids 364, 404, 442, 1862, 1914, 2004, 2529 |
| remodel_year_at_floor: YearRemodAdd = 1950 for a house built before 1950 (recording floor, DOC-02 §10.5). | confirmed | E-11 | 339 row(s) |
| Rare categories (< 1% of development rows) | confirmed | E-20 | 104 levels in 34 features |
| Dictionary-versus-file spellings | confirmed | E-02 | Allowed lists include the file spellings; values are not normalised |
| Register discrepancy P-44: Strong right skew in LotArea, GrLivArea, MasVnrArea, MiscVal, PoolArea, LowQualFinSF, 3SsnPorch | not confirmed | E-13 | LotArea 12.9, GrLivArea 0.9, MasVnrArea 2.4, MiscVal 21.4, PoolArea 18.2, LowQualFinSF 12.8, 3SsnPorch 11.3 |
